# Aurora Bay FAQ RAG Chatbot

This notebook demonstrates a **Retrieval-Augmented Generation (RAG)** system that:
1. Setup and Dependencies
2. Project Configuration
3. Create BigQuery Dataset
4. Load FAQ Data from GCS into BigQuery
5. Create BigQuery connection + grant IAM role
6. Create the BigQuery ML Remote Embedding Model
7. Embed user_question and return the top-k most relevant FAQ rows and perform smoke test
8. Gemini Response Generator
9. End-to-End RAG Chatbot Function
10. Demo: Ask the Chatbot
11. Interactive Chat Loop




## 1. Setup and Dependencies

Install and import the required libraries.

In [68]:
%pip install --quiet google-cloud-bigquery google-cloud-aiplatform

In [69]:
import json
import subprocess
import textwrap

from google.cloud import bigquery
import vertexai
from vertexai.generative_models import GenerativeModel

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Project Configuration

Set up variables to use all through notebook

In [70]:
PROJECT_ID = "qwiklabs-gcp-00-05feec5c5416"
REGION     = "us-central1"       # Vertex AI / BigQuery ML region

DATASET_ID      = "aurora_bay"
RAW_TABLE_ID    = "faqs"                 # table for raw FAQ data
EMBED_TABLE_ID  = "faqs_with_embeddings" # table for FAQs + embedding vectors
CONNECTION_NAME = "vertex-ai"            # BigQuery Cloud Resource Connection name
EMBED_MODEL_ID  = "embedding_model"      # BigQuery ML remote model alias

GCS_SOURCE = "gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv"  #Instructor provided data

# Fully-qualified BigQuery identifiers used in SQL statements
RAW_TABLE_REF   = f"`{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE_ID}`"
EMBED_TABLE_REF = f"`{PROJECT_ID}.{DATASET_ID}.{EMBED_TABLE_ID}`"
MODEL_REF       = f"`{PROJECT_ID}.{DATASET_ID}.{EMBED_MODEL_ID}`"
CONN_REF        = f"`{PROJECT_ID}.{REGION}.{CONNECTION_NAME}`"

# Initialize the BigQuery and Vertex AI clients
bq_client = bigquery.Client(project=PROJECT_ID)
vertexai.init(project=PROJECT_ID, location=REGION)

print(f"Project : {PROJECT_ID}")
print(f"Dataset : {DATASET_ID}")
print(f"Region  : {REGION}")

Project : qwiklabs-gcp-00-05feec5c5416
Dataset : aurora_bay
Region  : us-central1


## 3. Create BigQuery Dataset

Create the `aurora_bay` dataset if it does not already exist.
This dataset is the container for all tables and ML models used in this notebook.

In [71]:
def create_dataset_if_missing(client, project, dataset, location):
    """Create a BigQuery dataset in the given location, skipping if it already exists."""
    dataset_ref = bigquery.Dataset(f"{project}.{dataset}")
    dataset_ref.location = location
    client.create_dataset(dataset_ref, exists_ok=True)
    print(f"Dataset '{dataset}' is ready.")

create_dataset_if_missing(bq_client, PROJECT_ID, DATASET_ID, REGION)

Dataset 'aurora_bay' is ready.


## 4. Load FAQ Data from GCS into BigQuery

Copy the CSV directly from Google Cloud Storage
into the `faqs` table. The `skip_leading_rows = 1` option drops the CSV header row.
Test that the load happened
Identify the column names and set as variables to use later

In [72]:
load_sql = f"""
LOAD DATA OVERWRITE {RAW_TABLE_REF}
(question STRING, answer STRING)
FROM FILES (
    format            = 'CSV',
    uris              = ['{GCS_SOURCE}'],
    skip_leading_rows = 1
);
"""

print("Loading FAQ data from GCS into BigQuery...")
bq_client.query(load_sql).result()

row_count = next(bq_client.query(f"SELECT COUNT(*) AS n FROM {RAW_TABLE_REF}").result()).n
print(f"Done. {row_count} rows loaded into '{RAW_TABLE_ID}'.")

Loading FAQ data from GCS into BigQuery...
Done. 50 rows loaded into 'faqs'.


In [73]:
# Preview the first five rows to confirm the schema
bq_client.query(f"SELECT * FROM {RAW_TABLE_REF} LIMIT 5").result().to_dataframe()

,question,answer
0,When was Aurora Bay founded?,Aurora Bay was founded in 1901 by a group of f...
1,What is the population of Aurora Bay?,Aurora Bay has a population of approximately 3...
2,Where is the Aurora Bay Town Hall located?,The Town Hall is located at 100 Harbor View Ro...
3,Who is the current mayor of Aurora Bay?,"The current mayor is Linda Greenwood, elected ..."
4,What are the primary industries in Aurora Bay?,The primary industries include commercial fish...


In [74]:
table = bq_client.get_table(f"{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE_ID}")
column_names = [field.name for field in table.schema]
print("Columns in the FAQ table:", column_names)

QUESTION_COL = "question"
ANSWER_COL   = "answer"

print(f"Question column : {QUESTION_COL}")
print(f"Answer column   : {ANSWER_COL}")

Columns in the FAQ table: ['question', 'answer']
Question column : question
Answer column   : answer


## 5. Create BigQuery connection + grant IAM role
In the online classes this happened in the GUI

In [76]:
# Step 1: Create the Cloud Resource Connection (idempotent)
create_result = subprocess.run(
    [
        "bq", "mk", "--connection",
        f"--project_id={PROJECT_ID}",
        f"--location={REGION}",
        "--connection_type=CLOUD_RESOURCE",
        CONNECTION_NAME,
    ],
    capture_output=True,
    text=True,
)

print("Return code:", create_result.returncode)
print("STDOUT:", create_result.stdout)
print("STDERR:", create_result.stderr)

already_exists = (
    "already exists" in create_result.stdout.lower() or
    "already exists" in create_result.stderr.lower()
)

if create_result.returncode == 0 or already_exists:
    print(f"Connection '{CONNECTION_NAME}' is ready.")
else:
    raise RuntimeError("Failed to create BigQuery connection. See output above.")

# Step 2: Retrieve the auto-generated service account for this connection
show_result = subprocess.run(
    [
        "bq", "show", "--connection",
        f"--project_id={PROJECT_ID}",
        f"--location={REGION}",
        "--format=json",
        CONNECTION_NAME,
    ],
    capture_output=True,
    text=True,
)

if show_result.returncode != 0:
    raise RuntimeError(f"Failed to describe connection: {show_result.stderr}")

conn_info = json.loads(show_result.stdout)
service_account = conn_info.get("cloudResource", {}).get("serviceAccountId", "")

if not service_account:
    raise RuntimeError("Could not find service account in connection info.")

print(f"Service account: {service_account}")

# Step 3: Grant the Vertex AI User role to the connection service account
grant_result = subprocess.run(
    [
        "gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
        f"--member=serviceAccount:{service_account}",
        "--role=roles/aiplatform.user",
        "--condition=None",
    ],
    capture_output=True,
    text=True,
)

if grant_result.returncode == 0:
    print(f"Granted 'Vertex AI User' role to {service_account}")
else:
    print("STDERR:", grant_result.stderr)
    raise RuntimeError("Failed to grant IAM role. Check that your account has 'Project IAM Admin' permissions.")

Return code: 2
STDOUT: BigQuery error in mk operation: Already Exists: Connection
projects/709587511829/locations/us-central1/connections/vertex-ai

STDERR: 
Connection 'vertex-ai' is ready.
Service account: bqcx-709587511829-8d9p@gcp-sa-bigquery-condel.iam.gserviceaccount.com
Granted 'Vertex AI User' role to bqcx-709587511829-8d9p@gcp-sa-bigquery-condel.iam.gserviceaccount.com


## 6. Create the BigQuery ML Remote Embedding Model

Register a BigQuery ML **remote model** that points to
Vertex AI's `text-embedding-004` endpoint.

Generate embeddings and store in BigQuery

In class this happened in BigQuery GUI


In [77]:
create_model_sql = f"""
CREATE OR REPLACE MODEL {MODEL_REF}
  REMOTE WITH CONNECTION {CONN_REF}
  OPTIONS (
    endpoint = 'text-embedding-004'
  );
"""

print("Creating BigQuery ML remote embedding model...")
bq_client.query(create_model_sql).result()
print(f"Model '{EMBED_MODEL_ID}' created successfully.")

Creating BigQuery ML remote embedding model...
Model 'embedding_model' created successfully.


In [78]:
if not QUESTION_COL or not ANSWER_COL:
    raise RuntimeError("QUESTION_COL or ANSWER_COL not set. Run last part of step 4 first.")

print(f"Using columns: question='{QUESTION_COL}', answer='{ANSWER_COL}'")

generate_embeddings_sql = f"""
CREATE OR REPLACE TABLE {EMBED_TABLE_REF} AS

SELECT
    * EXCEPT(content, ml_generate_embedding_status, ml_generate_embedding_result),
    ml_generate_embedding_result AS embedding
FROM
    ML.GENERATE_EMBEDDING(
        MODEL {MODEL_REF},
        (
            SELECT
                *,
                CONCAT('Question: ', `{QUESTION_COL}`, ' Answer: ', `{ANSWER_COL}`) AS content
            FROM {RAW_TABLE_REF}
        ),
        STRUCT(TRUE AS flatten_json_output)
    );
"""

print("Generating embeddings -- this may take a minute...")
bq_client.query(generate_embeddings_sql).result()

embed_count = next(bq_client.query(f"SELECT COUNT(*) AS n FROM {EMBED_TABLE_REF}").result()).n
print(f"Done. {embed_count} rows with embeddings stored in '{EMBED_TABLE_ID}'.")

Using columns: question='question', answer='answer'
Generating embeddings -- this may take a minute...
Done. 50 rows with embeddings stored in 'faqs_with_embeddings'.


In [79]:
# Verify: confirm the embedding column exists and check its dimensionality
# Note: Ensure the cell above has finished running successfully before executing this.
sample_sql = f"""
SELECT
    `{QUESTION_COL}`,
    ARRAY_LENGTH(embedding) AS embedding_dims,
    embedding[OFFSET(0)]    AS dim_0,
    embedding[OFFSET(1)]    AS dim_1
FROM {EMBED_TABLE_REF}
LIMIT 3
"""
bq_client.query(sample_sql).result().to_dataframe()

,question,embedding_dims,dim_0,dim_1
0,When was Aurora Bay founded?,768,-0.003052,0.069000
1,Who is the current mayor of Aurora Bay?,768,0.048706,0.028038
2,What are the primary industries in Aurora Bay?,768,0.004998,0.068122


## 7. Embed user_question and return the top-k most relevant FAQ rows and  perform smoke test

In [80]:
def search_faqs(user_question, top_k=5):
    """
    Embed user_question and return the top-k most relevant FAQ rows
    from BigQuery using vector similarity search.

    Parameters
    ----------
    user_question : str   The raw question from the user.
    top_k         : int   Number of FAQ records to retrieve (default 5).

    Returns
    -------
    list of dict  Each dict has 'question', 'answer', and 'distance' keys.
    """
    # Escape single quotes to prevent SQL injection
    safe_question = user_question.replace("'", "''")

    vector_search_sql = f"""
    SELECT
        base.question,
        base.answer,
        distance
    FROM
        VECTOR_SEARCH(
            TABLE {EMBED_TABLE_REF},
            'embedding',
            (
                -- Embed the user question using the same model
                SELECT ml_generate_embedding_result AS embedding
                FROM ML.GENERATE_EMBEDDING(
                    MODEL {MODEL_REF},
                    (SELECT '{safe_question}' AS content),
                    STRUCT(TRUE AS flatten_json_output)
                )
            ),
            top_k         => {top_k},
            distance_type => 'COSINE'
        )
    ORDER BY distance ASC;
    """

    rows = bq_client.query(vector_search_sql).result()
    return [
        {"question": r.question, "answer": r.answer, "distance": r.distance}
        for r in rows
    ]


# Smoke test
test_results = search_faqs("What is the weather like in Aurora Bay?")
for i, r in enumerate(test_results, 1):
    print(f"{i}. [dist={r['distance']:.4f}] {r['question']}")

1. [dist=0.1218] What is the average temperature range in Aurora Bay?
2. [dist=0.2281] What is the population of Aurora Bay?
3. [dist=0.2338] What is the best way to travel to Aurora Bay?
4. [dist=0.2707] What types of recreation are available in Aurora Bay?
5. [dist=0.2749] Are there any hotels or lodging options in Aurora Bay?


## 8. Gemini Response Generator

Once we have the top-K FAQ matches, we build a **context block** and inject it into a prompt for Gemini.
This is the core of the RAG pattern:
- **Retrieval** — vector search fetches the most relevant FAQs
- **Augmentation** — retrieved text is inserted into the prompt as grounding context
- **Generation** — Gemini synthesizes a natural-language answer based only on that context,
  preventing hallucination of facts not in the proprietary data

In [81]:
gemini_model = GenerativeModel("gemini-2.5-flash")

def build_prompt(user_question, retrieved_faqs):
    """
    Construct the RAG prompt by prepending the retrieved FAQ context
    to the user question. The system instruction tells Gemini to answer
    only from the provided context and admit when it does not know.
    """
    context_block = "\n\n".join(
        f"Q: {faq['question']}\nA: {faq['answer']}"
        for faq in retrieved_faqs
    )

    return textwrap.dedent(f"""
        You are a helpful assistant for the town of Aurora Bay, Alaska.
        Answer the user's question using ONLY the official FAQ information provided below.
        If the answer is not contained in the FAQs, say so honestly -- do not guess.

        === AURORA BAY OFFICIAL FAQs ===
        {context_block}
        === END OF FAQs ===

        User question: {user_question}

        Provide a clear, concise, and friendly answer.
    """).strip()

def ask_gemini(prompt):
    """Send the prompt to Gemini and return the response text."""
    response = gemini_model.generate_content(prompt)
    return response.text

print("Gemini model updated (gemini-2.5-flash).")

Gemini model updated (gemini-2.5-flash).


/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


## 9. End-to-End RAG Chatbot Function

The `chat` function ties everything together:
1. Vector-search BigQuery for the most relevant FAQs
2. Build a grounded prompt from the retrieved context
3. Call Gemini and return the answer

In [82]:
def chat(user_question, top_k=5, verbose=False):
    """
    End-to-end RAG pipeline for the Aurora Bay FAQ chatbot.

    Parameters
    ----------
    user_question : str   Plain-English question from the user.
    top_k         : int   Number of FAQs to retrieve for context (default 5).
    verbose       : bool  If True, print the retrieved FAQs before the answer.

    Returns
    -------
    str  Gemini's grounded answer.
    """
    retrieved = search_faqs(user_question, top_k=top_k)

    if verbose:
        print("Retrieved FAQs:")
        for i, faq in enumerate(retrieved, 1):
            print(f"  {i}. [cosine dist={faq['distance']:.4f}] {faq['question']}")
        print()

    prompt = build_prompt(user_question, retrieved)
    return ask_gemini(prompt)

## 10. Demo: Ask the Chatbot

Run the cells below to ask the Aurora Bay FAQ chatbot various questions.
Set `verbose=True` to see which FAQ records were retrieved before Gemini generated the answer.

In [83]:
question = "How do I get a fishing license in Aurora Bay?"
answer = chat(question, verbose=True)
print(f"Q: {question}\n")
print(f"A: {answer}")

Retrieved FAQs:
  1. [cosine dist=0.1964] Where can I find information about local fishing regulations?
  2. [cosine dist=0.2360] How can I apply for a business license in Aurora Bay?
  3. [cosine dist=0.2765] What is the population of Aurora Bay?
  4. [cosine dist=0.2837] Does Aurora Bay have a police department?
  5. [cosine dist=0.2924] Does Aurora Bay have a public library?

Q: How do I get a fishing license in Aurora Bay?

A: I'm sorry, but the provided FAQs do not contain information on how to get a fishing license in Aurora Bay. The FAQs only mention where to find information about fishing regulations.


In [85]:
question = "What are the garbage collection days?"
answer = chat(question, verbose=True)
print(f"Q: {question}\n")
print(f"A: {answer}")

Retrieved FAQs:
  1. [cosine dist=0.1976] What is the procedure for trash collection?
  2. [cosine dist=0.4600] Are there specific quiet hours or noise ordinances?
  3. [cosine dist=0.4809] Does Aurora Bay have a recycling program?
  4. [cosine dist=0.4877] When are the town council meetings held?
  5. [cosine dist=0.5263] When does the local fishermen’s market usually take place?

Q: What are the garbage collection days?

A: Trash is collected once a week based on your assigned neighborhood schedule. The FAQs do not specify the exact garbage collection days for individual neighborhoods.


In [86]:
question = "Is there a public library in Aurora Bay?"
answer = chat(question, verbose=True)
print(f"Q: {question}\n")
print(f"A: {answer}")

Retrieved FAQs:
  1. [cosine dist=0.0748] Does Aurora Bay have a public library?
  2. [cosine dist=0.1686] What are the operating hours of the Aurora Bay Public Library?
  3. [cosine dist=0.2215] Does Aurora Bay have a local historical museum?
  4. [cosine dist=0.2267] Does Aurora Bay have a police department?
  5. [cosine dist=0.2333] Is there a local hospital in Aurora Bay?

Q: Is there a public library in Aurora Bay?

A: Yes, Aurora Bay has a public library. The Aurora Bay Public Library is located on Main Street, next to the town's post office.


## 11. Interactive Chat Loop

Run this cell to start an interactive session in the notebook. Type `quit` or `exit` to stop.

In [87]:
print("Aurora Bay FAQ Chatbot -- type 'quit' to exit")
print("=" * 50)

while True:
    user_input = input("\nYou: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit"):
        print("Goodbye!")
        break
    response = chat(user_input)
    print(f"\nBot: {response}")
    print("-" * 50)

Aurora Bay FAQ Chatbot -- type 'quit' to exit

You: What is it like in Albany NY today?

Bot: I'm sorry, but the provided FAQs only contain information about Aurora Bay, Alaska. I do not have any information about Albany, NY.
--------------------------------------------------

You: quit
Goodbye!
